Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\24pasos_mlp_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [ ]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 24
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 7)
Dimensiones de Y: (43765, 1)


In [9]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [10]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (43765, 84)


Se dividen nuevamente los conjuntos de datos

In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 84)
Las dimensiones de testX son:  (8797, 84)
Las dimensiones de valX son:  (4333, 84)


In [12]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [13]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])

    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

49/49 - 7s - 139ms/step - ia: 0.2487 - loss: 1.4326 - mae: 0.9227 - rmse: 1.1960 - smape: 1.5000 - val_ia: 0.2623 - val_loss: 0.5937 - val_mae: 0.6233 - val_rmse: 0.7380 - val_smape: 1.7768

Epoch 2/128                                           

49/49 - 0s - 4ms/step - ia: 0.2582 - loss: 1.3502 - mae: 0.8879 - rmse: 1.1578 - smape: 1.4774 - val_ia: 0.2613 - val_loss: 0.5737 - val_mae: 0.6045 - val_rmse: 0.7214 - val_smape: 1.8367

Epoch 3/128                                           

49/49 - 0s - 6ms/step - ia: 0.2541 - loss: 1.3777 - mae: 0.8862 - rmse: 1.1720 - smape: 1.4905 - val_ia: 0.2604 - val_loss: 0.5593 - val_mae: 0.5901 - val_rmse: 0.7087 - val_smape: 1.9388

Epoch 4/128                                           

49/49 - 0s - 4ms/step - ia: 0.2565 - loss: 1.3372 - mae: 0.8712 - rmse: 1.1400 - smape: 1.4880 - val_ia: 0.2618 - val_loss: 0.5553 - val_mae: 0.5863 - val_rmse: 0.7053 - val_smape: 1.9431

Epoch 5/128       

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



385/385 - 4s - 10ms/step - ia: 0.4124 - loss: 0.9793 - mae: 0.7340 - rmse: 0.9584 - smape: 1.2386 - val_ia: 0.2556 - val_loss: 0.4993 - val_mae: 0.5283 - val_rmse: 0.6051 - val_smape: 1.2067

Epoch 2/128                                                                     

385/385 - 1s - 3ms/step - ia: 0.4159 - loss: 0.9198 - mae: 0.7077 - rmse: 0.9271 - smape: 1.2278 - val_ia: 0.2466 - val_loss: 0.5201 - val_mae: 0.5468 - val_rmse: 0.6179 - val_smape: 1.2311

Epoch 3/128                                                                     

385/385 - 1s - 2ms/step - ia: 0.4216 - loss: 0.9079 - mae: 0.7041 - rmse: 0.9236 - smape: 1.2282 - val_ia: 0.2436 - val_loss: 0.5125 - val_mae: 0.5474 - val_rmse: 0.6135 - val_smape: 1.3077

Epoch 4/128                                                                     

385/385 - 1s - 2ms/step - ia: 0.4166 - loss: 0.8999 - mae: 0.7031 - rmse: 0.9182 - smape: 1.2286 - val_ia: 0.2324 - val_loss: 0.5172 - val_mae: 0.5571 - val_rmse: 0.6218 - val_smap

...

385/385 - 4s - 10ms/step - ia: 0.3111 - loss: 1.4054 - mae: 0.9056 - rmse: 1.1543 - smape: 1.4071 - val_ia: 0.2502 - val_loss: 0.4690 - val_mae: 0.5010 - val_rmse: 0.5573 - val_smape: 1.2016

Epoch 2/128                                                                     

385/385 - 1s - 3ms/step - ia: 0.3437 - loss: 1.2747 - mae: 0.8527 - rmse: 1.0968 - smape: 1.3409 - val_ia: 0.2565 - val_loss: 0.4544 - val_mae: 0.4980 - val_rmse: 0.5546 - val_smape: 1.2229

Epoch 3/128                                                                     

385/385 - 1s - 3ms/step - ia: 0.3565 - loss: 1.1767 - mae: 0.8279 - rmse: 1.0586 - smape: 1.3328 - val_ia: 0.2577 - val_loss: 0.4521 - val_mae: 0.4942 - val_rmse: 0.5515 - val_smape: 1.2007

Epoch 4/128                                                                     

385/385 - 1s - 2ms/step - ia: 0.3732 - loss: 1.0979 - mae: 0.7962 - rmse: 1.0212 - smape: 1.3024 - val_ia: 0.2570 - val_loss: 0.4510 - val_mae: 0.4971 - val_rmse: 0.5542 - val_smap

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



770/770 - 4s - 5ms/step - ia: 0.2062 - loss: 9.0031 - mae: 1.9989 - rmse: 2.7338 - smape: 1.6638 - val_ia: 0.1480 - val_loss: 1.5319 - val_mae: 1.0191 - val_rmse: 1.0982 - val_smape: 1.6568

Epoch 2/128                                                                      

770/770 - 2s - 2ms/step - ia: 0.2139 - loss: 7.9179 - mae: 1.8916 - rmse: 2.5636 - smape: 1.6462 - val_ia: 0.1492 - val_loss: 1.4989 - val_mae: 1.0071 - val_rmse: 1.0857 - val_smape: 1.6555

Epoch 3/128                                                                      

770/770 - 1s - 2ms/step - ia: 0.2085 - loss: 8.1249 - mae: 1.9186 - rmse: 2.6206 - smape: 1.6489 - val_ia: 0.1503 - val_loss: 1.4678 - val_mae: 0.9959 - val_rmse: 1.0740 - val_smape: 1.6544

Epoch 4/128                                                                      

770/770 - 3s - 3ms/step - ia: 0.2125 - loss: 8.4915 - mae: 1.9268 - rmse: 2.6328 - smape: 1.6554 - val_ia: 0.1513 - val_loss: 1.4361 - val_mae: 0.9842 - val_rmse: 1.0618 - val_sm

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



49/49 - 2s - 34ms/step - ia: 0.3240 - loss: 1.1457 - mae: 0.8069 - rmse: 1.0612 - smape: 1.3725 - val_ia: 0.3072 - val_loss: 0.4582 - val_mae: 0.5028 - val_rmse: 0.6383 - val_smape: 1.2384

Epoch 2/128                                                                      

49/49 - 0s - 3ms/step - ia: 0.3876 - loss: 1.0330 - mae: 0.7655 - rmse: 1.0193 - smape: 1.2865 - val_ia: 0.3146 - val_loss: 0.4653 - val_mae: 0.5074 - val_rmse: 0.6433 - val_smape: 1.2445

Epoch 3/128                                                                      

49/49 - 0s - 3ms/step - ia: 0.4095 - loss: 0.9942 - mae: 0.7518 - rmse: 0.9913 - smape: 1.2646 - val_ia: 0.3161 - val_loss: 0.4660 - val_mae: 0.5108 - val_rmse: 0.6460 - val_smape: 1.2443

Epoch 4/128                                                                      

49/49 - 0s - 3ms/step - ia: 0.4143 - loss: 0.9962 - mae: 0.7499 - rmse: 0.9859 - smape: 1.2485 - val_ia: 0.3153 - val_loss: 0.4671 - val_mae: 0.5110 - val_rmse: 0.6462 - val_smape: 1.

In [16]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}


In [17]:
#{'activation': 3, 'batch': 4, 'dropout': 0.4, 'layers': 1.0, 'learning_rate': 0.009726326397617627, 'units': 3}